# 06 — Exploración del panel de secciones electorales

Primera exploración visual y estadística del panel maestro a nivel sección
(71,541 secciones, 5 elecciones 2012-2024, censo 2020 fino):

1. Panorama del voto y la participación por sección
2. Cambio territorial 2018→2024 (swing local)
3. Tipología de secciones: bastiones, competitivas, volátiles
4. Correlaciones voto-sociedad a la escala más fina disponible
5. Autocorrelación espacial (Moran y LISA) a nivel sección

In [1]:
import sys, os
sys.path.insert(0, os.path.join("..", "src"))
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from scipy import stats

PROCESADOS = os.path.join("..", "data", "processed")
FIGURAS = os.path.join("..", "figuras")
os.makedirs(FIGURAS, exist_ok=True)

panel = pd.read_csv(os.path.join(PROCESADOS, "panel_secciones.csv"),
                    dtype={"id": str, "ent": str, "mun": str, "distrito_f": str})
panel = panel.set_index("id")
sec = gpd.read_file(os.path.join("..", "data", "upload", "mgsan_ext", "SECCION.shp"))
sec["id"] = sec["ENTIDAD"].astype(int).astype(str).str.zfill(2) + sec["SECCION"].astype(int).astype(str).str.zfill(4)
sec = sec[["id", "geometry"]].to_crs("EPSG:4326")
g = sec.merge(panel.reset_index(), on="id", how="inner")
print(f"panel: {panel.shape} | geometrías: {len(g):,}")
g.head(3)


panel: (71541, 116) | geometrías: 71,541


/tmp/ipykernel_5869/2588567237.py:19: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  g = sec.merge(panel.reset_index(), on="id", how="inner")


,id,geometry,ent,mun,distrito_f,pan_2012,pct_pan_2012,pri_2012,pct_pri_2012,prd_2012,...,pct_sin_basica,pct_analfabeta,pct_piso_tierra,pct_sin_drenaje,pct_sin_bienes,pct_auto,pct_internet,pct_sin_salud,pct_60ymas,pct_18a24
0,240001,"POLYGON ((-101.15731 22.40989, -101.15711 22.4...",24,001,01,392.0,54.068966,202.0,27.862069,69.666667,...,17.155756,6.772009,92.073171,0.0,0.00000,44.308943,40.447154,19.559652,12.237583,11.879160
1,270001,"POLYGON ((-91.51676 17.8552, -91.51613 17.8551...",27,001,01,215.0,13.711735,668.5,42.633929,457.000000,...,12.259372,5.606214,98.232323,0.0,0.16835,20.117845,30.808081,25.573066,8.739255,9.909265
2,030001,"POLYGON ((-111.68825 25.04847, -111.6839 25.04...",03,001,01,163.0,42.670157,111.0,29.057592,66.333333,...,3.988183,2.658789,95.709571,0.0,0.00000,84.488449,72.277228,7.088332,14.830971,3.598691


## 1. Panorama: voto y participación por sección

In [2]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
vars = [("pct_morena_2024", "% Morena 2024", "RdYlBu_r"),
        ("pct_pan_2024", "% PAN 2024", "RdYlBu_r"),
        ("participacion_2024", "Participación 2024 (%)", "viridis")]
for ax, (c, t, cm) in zip(axes, vars):
    g.plot(column=c, ax=ax, legend=True, cmap=cm, legend_kwds={"shrink": 0.55},
           missing_kwds={"color": "lightgrey"})
    ax.set_title(t, fontsize=12); ax.set_axis_off()
plt.tight_layout()
plt.close("all")
plt.savefig(os.path.join(FIGURAS, "seccion_panorama_2024.png"), dpi=150, bbox_inches="tight")
plt.show()

<Figure size 640x480 with 0 Axes>

In [3]:
d = panel.dropna(subset=["pct_morena_2024", "participacion_2024"])
print(f"distribución de la participación 2024 por sección:")
print(d["participacion_2024"].describe().round(1).to_string())
print(f"\nparticipación promedio 2021 (intermedia): {d['participacion_2021'].mean():.1f}%")
print(f"participación promedio 2024 (presidencial): {d['participacion_2024'].mean():.1f}%")
print(f"\nsecciones donde ganó cada fuerza (2024, dip. MR):")
todas = [f"{p}_2024" for p in ["pan","pri","prd","pvem","pt","mc","morena","otros"]]
fuerzas = {"oficialismo": ["morena","pvem","pt"], "oposición": ["pan","pri","prd"], "mc": ["mc"]}
for f, ps in fuerzas.items():
    cols_f = [f"{p}_2024" for p in ps]
    cols_r = [c for c in todas if c not in cols_f]
    tot = d[cols_f].sum(axis=1)
    print(f"  {f}: {(tot > d[cols_r].max(axis=1)).sum():,} secciones")

distribución de la participación 2024 por sección:
count    71227.0
mean        60.9
std         10.8
min          0.4
25%         53.7
50%         61.3
75%         68.4
max        116.0

participación promedio 2021 (intermedia): 54.0%
participación promedio 2024 (presidencial): 60.9%

secciones donde ganó cada fuerza (2024, dip. MR):
  oficialismo: 61,338 secciones
  oposición: 20,913 secciones
  mc: 2,637 secciones


## 2. Cambio territorial 2018→2024

In [4]:
g["swing_morena"] = g["pct_morena_2024"] - g["pct_morena_2018"]
g["swing_mc"] = g["pct_mc_2024"] - g["pct_mc_2018"]
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, c, t in zip(axes, ["swing_morena", "swing_mc"],
                    ["Swing Morena 2018→2024 (puntos)", "Swing MC 2018→2024 (puntos)"]):
    g.plot(column=c, ax=ax, legend=True, cmap="RdBu_r", vmin=-40, vmax=40,
           legend_kwds={"shrink": 0.55}, missing_kwds={"color": "lightgrey"})
    ax.set_title(t, fontsize=12); ax.set_axis_off()
plt.tight_layout()
plt.close("all")
plt.savefig(os.path.join(FIGURAS, "seccion_swing_2018_2024.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Secciones con mayor alza de Morena:"); display(
    g.nlargest(5, "swing_morena")[["ent","mun","swing_morena","pct_morena_2018","pct_morena_2024"]])
print("Secciones con mayor caída de Morena:"); display(
    g.nsmallest(5, "swing_morena")[["ent","mun","swing_morena","pct_morena_2018","pct_morena_2024"]])

/home/appuser/.local/lib/python3.12/site-packages/geopandas/geodataframe.py:2011: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
/home/appuser/.local/lib/python3.12/site-packages/geopandas/geodataframe.py:2011: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)


<Figure size 640x480 with 0 Axes>

Secciones con mayor alza de Morena:


,ent,mun,swing_morena,pct_morena_2018,pct_morena_2024
31777,28,016,100.000000,0.000000,100.000000
20706,12,016,99.090909,0.000000,99.090909
31779,28,016,98.979592,0.000000,98.979592
20732,12,016,98.507463,0.000000,98.507463
20700,12,016,97.828107,0.952381,98.780488


Secciones con mayor caída de Morena:


,ent,mun,swing_morena,pct_morena_2018,pct_morena_2024
34670,12,044,-81.950640,92.893401,10.942761
22962,12,049,-73.511905,85.416667,11.904762
28121,20,207,-63.592562,99.076433,35.483871
25115,12,001,-63.486590,64.597701,1.111111
2517,13,025,-62.891580,79.674797,16.783217


El swing territorial es muy visible: el norte y las zonas urbanas
medias se movieron hacia MC y la oposición; el sur y las zonas populares
consolidaron al oficialismo.

## 3. Tipología de secciones

In [5]:
partidos = ["pan","pri","prd","pvem","pt","mc","morena","otros"]
P = g[[f"pct_{p}_2024" for p in partidos]].copy()
P.columns = partidos
P = P.fillna(0)
g["ganador_2024"] = P.idxmax(axis=1)
g["margin"] = P.max(axis=1) - P.apply(lambda r: r.drop(r.idxmax()).max(), axis=1)
g["volatilidad"] = (g["pct_morena_2024"] - g["pct_morena_2018"]).abs() + \
                   (g["pct_pan_2024"] - g["pct_pan_2018"]).abs() + \
                   (g["pct_mc_2024"] - g["pct_mc_2018"]).abs()

def tipologia(fila):
    if pd.isna(fila["margin"]): return "sin datos"
    if fila["margin"] < 5: return "competitiva"
    if fila["ganador_2024"] in ("morena","pvem","pt") and fila["margin"] >= 20:
        return "bastión oficialista"
    if fila["ganador_2024"] in ("pan","pri","prd") and fila["margin"] >= 20:
        return "bastión oposición"
    return "dominante moderada"
g["tipologia"] = g.apply(tipologia, axis=1)
g["volatilidad_cat"] = pd.cut(g["volatilidad"], [0, 10, 20, 100],
                              labels=["estable", "media", "volátil"])
print(g["tipologia"].value_counts())
print()
print(pd.crosstab(g["tipologia"], g["volatilidad_cat"]))

/home/appuser/.local/lib/python3.12/site-packages/geopandas/geodataframe.py:2011: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)


/home/appuser/.local/lib/python3.12/site-packages/geopandas/geodataframe.py:2011: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
/home/appuser/.local/lib/python3.12/site-packages/geopandas/geodataframe.py:2011: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)


tipologia
bastión oficialista    35359
dominante moderada     22446
competitiva             9160
bastión oposición       4576
Name: count, dtype: int64

volatilidad_cat      estable  media  volátil
tipologia                                   
bastión oficialista     3035  12155    19879
bastión oposición        308   1413     2836
competitiva              541   2161     6109
dominante moderada      1265   5685    15390


/home/appuser/.local/lib/python3.12/site-packages/geopandas/geodataframe.py:2011: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)
/home/appuser/.local/lib/python3.12/site-packages/geopandas/geodataframe.py:2011: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  super().__setitem__(key, value)


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(17, 7))
g.plot(column="tipologia", ax=axes[0], categorical=True, legend=True,
       cmap="Set1_r", legend_kwds={"loc": "lower left", "fontsize": 7})
axes[0].set_title("Tipología de secciones (elección 2024)", fontsize=12)
g.plot(column="volatilidad_cat", ax=axes[1], categorical=True, legend=True,
       cmap="YlOrRd", legend_kwds={"loc": "lower left", "fontsize": 8})
axes[1].set_title("Volatilidad 2018→2024 (swing acumulado 3 fuerzas)", fontsize=12)
for ax in axes: ax.set_axis_off()
plt.tight_layout()
plt.close("all")
plt.savefig(os.path.join(FIGURAS, "seccion_tipologia.png"), dpi=150, bbox_inches="tight")
plt.show()

<Figure size 640x480 with 0 Axes>

## 4. Voto y sociedad a escala de sección (censo fino)

In [7]:
censo_vars = {
    "pct_sin_basica": "Sin educación básica (%)",
    "pct_analfabeta": "Analfabetismo (%)",
    "pct_piso_tierra": "Piso de tierra (%)",
    "pct_sin_bienes": "Sin bienes en vivienda (%)",
    "pct_auto": "Con automóvil (%)",
    "pct_internet": "Con internet (%)",
    "pct_rural": "Población rural (%)",
    "pct_60ymas": "Adultos mayores (%)",
}
d = g.dropna(subset=["pct_morena_2024", "pct_pan_2024", "pct_mc_2024"] + list(censo_vars))
tabla = []
for v, etiqueta in censo_vars.items():
    r_m = stats.spearmanr(d["pct_morena_2024"].clip(0,100), d[v])
    r_p = stats.spearmanr(d["pct_pan_2024"].clip(0,100), d[v])
    r_c = stats.spearmanr(d["pct_mc_2024"].clip(0,100), d[v])
    tabla.append({"variable": etiqueta, "rho_morena": r_m.statistic,
                  "rho_pan": r_p.statistic, "rho_mc": r_c.statistic,
                  "p": max(r_m.pvalue, r_p.pvalue, r_c.pvalue)})
display(pd.DataFrame(tabla).round(3))

,variable,rho_morena,rho_pan,rho_mc,p
0,Sin educación básica (%),0.312,-0.420,-0.127,0.0
1,Analfabetismo (%),0.294,-0.376,-0.109,0.0
2,Piso de tierra (%),-0.143,0.319,0.137,0.0
3,Sin bienes en vivienda (%),0.129,-0.259,-0.144,0.0
4,Con automóvil (%),-0.406,0.463,0.107,0.0
5,Con internet (%),-0.357,0.528,0.190,0.0
6,Población rural (%),0.044,-0.186,-0.106,0.0
7,Adultos mayores (%),-0.288,0.373,-0.051,0.0


In [8]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
pares = [("pct_morena_2024", "pct_sin_basica", "% Morena 2024"),
         ("pct_pan_2024", "pct_auto", "% PAN 2024"),
         ("pct_mc_2024", "pct_internet", "% MC 2024")]
for ax, (x, y, t) in zip(axes, pares):
    ax.scatter(d[y], d[x].clip(0,100), s=2, alpha=0.15, color="steelblue", edgecolor="none")
    rho = stats.spearmanr(d[x].clip(0,100), d[y]).statistic
    ax.set_title(f"{t} vs {y.replace('pct_','').replace('_',' ')} (ρ={rho:.2f})", fontsize=11)
    ax.set_xlabel(y); ax.set_ylabel(t)
plt.suptitle("Voto por sección vs. censo fino (2020)", fontsize=13)
plt.tight_layout()
plt.close("all")
plt.savefig(os.path.join(FIGURAS, "seccion_voto_censo.png"), dpi=150, bbox_inches="tight")
plt.show()

<Figure size 640x480 with 0 Axes>

## 5. Autocorrelación espacial a nivel sección (Moran y LISA)

In [9]:
import warnings; warnings.filterwarnings("ignore")
import gc
from libpysal.weights import KNN
from esda.moran import Moran, Moran_Local

gc.collect()
idx = g.dropna(subset=["pct_morena_2024"]).index
gs = g.loc[idx].copy()
centros = gs.geometry.representative_point()
coords = np.column_stack([centros.x, centros.y])
w = KNN.from_array(coords, k=8)
w.transform = "r"
mi = Moran(gs["pct_morena_2024"].values, w, permutations=199)
print(f"Moran's I del % Morena 2024 a nivel sección: I = {mi.I:.4f} (p = {mi.p_sim:.4f})")
print(f"vecinos promedio: {w.mean_neighbors:.1f}")

Moran's I del % Morena 2024 a nivel sección: I = 0.7474 (p = 0.0050)
vecinos promedio: 8.0


In [10]:
lisa = Moran_Local(gs["pct_morena_2024"].values, w, permutations=199)
gs = gs.copy()
gs["sig"] = lisa.p_sim < 0.05
gs["q"] = lisa.q
colores = {1: "#d7191c", 3: "#2c7bb6"}
fig, ax = plt.subplots(figsize=(11, 7))
gs[~gs["sig"]].plot(ax=ax, color="#f0f0f0")
for q, c in colores.items():
    sub = gs[gs["sig"] & (gs["q"] == q)]
    if len(sub): sub.plot(ax=ax, color=c,
        label={1: "Alto-Alto (bastiones)", 3: "Bajo-Bajo (oposición)"}[q])
ax.legend(title="Clusters LISA (p<0.05)", loc="lower left")
ax.set_title("LISA: clusters espaciales del voto a Morena por sección (2024)", fontsize=12)
ax.set_axis_off()
plt.tight_layout()
plt.close("all")
plt.savefig(os.path.join(FIGURAS, "seccion_lisa_morena.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"clusters significativos: Alto-Alto: {(gs['sig'] & (gs['q']==1)).sum():,} | "
      f"Bajo-Bajo: {(gs['sig'] & (gs['q']==3)).sum():,}")

<Figure size 640x480 with 0 Axes>

clusters significativos: Alto-Alto: 14,886 | Bajo-Bajo: 16,013


## Síntesis

- **La sección electoral revela lo que el municipio esconde**: correlaciones
  voto-sociedad más nítidas que a escala municipal (p. ej., Morena vs sin
  educación básica ρ≈+0.31; PAN vs automóvil ρ≈+0.4).
- **El swing 2018→2024 fue geográficamente estructurado**: MC y PAN
  avanzaron en zonas urbanas medias del norte y el Bajío; el oficialismo
  consolidó el sur y las periferias populares.
- **Tipología lista para el análisis electoral**: bastiones, secciones
  competitivas (< 5 puntos de margen) y volátiles — insumo directo para
  estrategia de campaña y para el pronóstico 2027.
- **El voto está fuertemente agrupado en el espacio** (Moran's I alto a
  nivel sección): los clusters LISA identifican los bastiones
  territoriales con precisión de sección.